In [214]:
import numpy as np
import pandas as pd
import tifffile
import os
from skimage.registration import phase_cross_correlation
from skimage.transform import estimate_transform, warp
import pickle
from scipy.ndimage import shift
from tqdm import tqdm
from skimage.util import img_as_uint

# phase cross correlation

In [2]:
def pad_or_crop_to_shape(image, target_shape):
    """Pad or crop image to match the target shape (centered)."""
    result = np.zeros(target_shape, dtype=image.dtype)

    # Compute the overlapping region
    min_shape = np.minimum(image.shape, target_shape)
    start_src = [(s - m) // 2 for s, m in zip(image.shape, min_shape)]
    start_dst = [(s - m) // 2 for s, m in zip(target_shape, min_shape)]

    # Copy over the overlapping region
    slices_src = tuple(slice(start, start + size) for start, size in zip(start_src, min_shape))
    slices_dst = tuple(slice(start, start + size) for start, size in zip(start_dst, min_shape))
    result[slices_dst] = image[slices_src]
    
    return result

In [112]:
in_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA+_83\01_ro60_lum_ro52\registration'

In [67]:
reference_im = tifffile.imread(os.path.join(in_dir, 'C1', 'C1_01_msg_141_ssa+_ro60_empty_ro52.tif'))
moving_im = tifffile.imread(os.path.join(in_dir, 'C1', 'C1_01_msg_141_ssa+_ro60_lum_ro52.tif'))

In [68]:
moving_im = pad_or_crop_to_shape(moving_im, reference_im.shape)
shift_estimation, error, diffphase = phase_cross_correlation(reference_im, moving_im, upsample_factor=100)

In [69]:
registered_image = shift(moving_im, shift_estimation, mode='constant', cval=0)

In [70]:
tifffile.imwrite(os.path.join(in_dir, 'C1_01_msg_141_ssa+_ro60_lum_ro52_registered.tif'), registered_image)

In [71]:
for c in tqdm(['C2', 'C3', 'C4', 'C5']):
    fn_l = os.listdir(os.path.join(in_dir, c))
    for fn in fn_l:
        if fn.endswith('.tif'):
            moving_im = tifffile.imread(os.path.join(in_dir, c, fn))
            moving_im = pad_or_crop_to_shape(moving_im, reference_im.shape)
            registered_image = shift(moving_im, shift_estimation, mode='constant', cval=0)
            tifffile.imwrite(os.path.join(in_dir, c, fn.replace('.tif', '_registered.tif')), registered_image)

100%|██████████| 4/4 [01:19<00:00, 19.83s/it]


# Mannual annotated matching points

In [301]:
im_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA-_7223\01_ro60_lum_ro52\registration'

In [302]:
station_im = tifffile.imread(os.path.join(im_dir, 'C1', 'C1_01_msg_7223_ssa-_ro60_empty_ro52.tif'))
shape = station_im.shape

In [303]:
if_ref = pd.read_csv(os.path.join(im_dir, 'if_ref.csv'))
lum_ref = pd.read_csv(os.path.join(im_dir, 'lum_ref.csv'))

In [304]:
tform = estimate_transform('affine',
                            np.stack((lum_ref['col'].tolist(), lum_ref['row'].tolist()), axis=1),
                            np.stack((if_ref['col'].tolist(), if_ref['row'].tolist()),axis=1))
with open(os.path.join(im_dir, 'transformation.pkl'), 'wb') as f:
        pickle.dump(tform, f)

In [305]:
for c in tqdm(['C1', 'C2', 'C3', 'C4', 'C5']):
    fn_l = os.listdir(os.path.join(im_dir, c))
    for fn in fn_l:
        if fn.endswith('.tif') and 'lum' in fn:
            moving_im = tifffile.imread(os.path.join(im_dir, c, fn))
            warped_image = warp(moving_im, tform.inverse, output_shape=shape)
            tifffile.imwrite(os.path.join(im_dir, c, fn.replace('.tif', '_registered.tif')), img_as_uint(warped_image))

100%|██████████| 5/5 [01:50<00:00, 22.07s/it]


# cropping of intact sections

In [313]:
# cropped section template matching
from skimage.feature import match_template

In [314]:
lum_registered_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA-_7223\01_ro60_lum_ro52\registration'
lum_cropped_dir = r'Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA-_7223\crop2'

In [315]:
cropping = pd.read_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20250123_ssa_scan\ssa-_7223\00_registration\crop2\crop2.csv', index_col=0)

In [316]:
row_min = int(cropping['row'].min())
row_max = int(cropping['row'].max())
col_min = int(cropping['col'].min())
col_max = int(cropping['col'].max())

In [317]:
channels_l = ['C1', 'C2', 'C3', 'C4', 'C5']

In [318]:
for channel in tqdm(channels_l):
    os.makedirs(os.path.join(lum_cropped_dir, channel), exist_ok=True)
    fn = os.listdir(os.path.join(lum_registered_dir, channel))
    for f in fn:
        if f.endswith('.tif') and 'registered' in f:
            im = tifffile.imread(os.path.join(lum_registered_dir, channel, f))
            im_cropped = im[row_min:row_max, col_min:col_max]
            tifffile.imwrite(os.path.join(lum_cropped_dir, channel, f.replace('.tif', '_cropped.tif')), im_cropped)

100%|██████████| 5/5 [00:08<00:00,  1.66s/it]


# LUM dot detection

In [262]:
from skimage.feature import peak_local_max
from skimage.filters import threshold_triangle
from skimage.util import img_as_float
import pandas as pd

In [331]:
im = tifffile.imread(r"Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA+_141\01_ro60_lum_ro52\registration\C3\C3_01_msg_141_ssa+_ro60_lum_ro52_registered.tif")

In [332]:
im = img_as_float(im)
thre = threshold_triangle(im)
dots = peak_local_max(im, min_distance=3, threshold_abs=thre, exclude_border=False)

In [333]:
dots_df = pd.DataFrame(np.array(dots), columns=['row','col'])

In [334]:
dots_df.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA+_141\dots_df.csv')

# count per cell

In [366]:
from skimage.measure import regionprops_table

In [367]:
dots_df = pd.read_csv(r"Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA+_141\dots_df.csv", index_col=0)
mask = tifffile.imread(r"Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA+_141\01_ro60_lum_ro52\registration\C1\C1_01_msg_141_ssa+_ro60_lum_ro52_registered_cp_masks.tif")

In [368]:
properties = regionprops_table(mask, properties=['label', 'area', 'centroid'])
dots_df['label'] = mask[dots_df['row'], dots_df['col']]
dots_df = dots_df[dots_df['label'] != 0]

In [369]:
dots_df.to_csv(r"Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA+_141\dots_df.csv")

In [370]:
# count number of dots per label
dots_count = dots_df['label'].value_counts().reset_index()
joined = pd.DataFrame(properties).merge(dots_count, left_on='label', right_on='label', how='left').fillna(0)
joined.rename(columns={'centroid-0':'row', 'centroid-1':'col'}, inplace=True)

In [371]:
joined.to_csv(r'Y:\coskun-lab2\Zhou\12_MSG\20250523_ro_lum\SSA+_141\counts_df.csv')